# What happened when I tried distillation?

I wanted a small model that reads a support request and predicts its intent. I also wanted to see whether a larger teacher could make that small model better. This notebook records the checks I found useful. Download the CSV from the Bitext Kaggle page linked in the README, save it as `data/raw/bitext/requests.csv`, then run `python train.py --evaluate-test` first.

## 1. Look at the actual data

I started with the request text and the intent supplied by Bitext. The intent is the answer the model tries to learn.

In [ ]:
from pathlib import Path
import csv

root = Path.cwd()
if not (root / "train.py").exists():
    root = root.parent
csv_path = root / "data/raw/bitext/requests.csv"
with csv_path.open(newline="", encoding="utf-8-sig") as file:
    rows = list(csv.DictReader(file))

print("Rows:", len(rows))
print("Columns:", list(rows[0]))
for row in rows[:3]:
    print(row["utterance"], "→", row["intent"])

The CSV is a set of authored support examples. It is useful for an experiment, but it is not a sample of live customer traffic. The code keeps Bitext’s 27 labels.

## 2. What the teacher gives the student

A classifier gives one score per intent. Bigger scores mean stronger guesses. Softmax turns scores into probabilities. Temperature spreads those probabilities out, so a student can also see which losing answers the teacher considered.

In [ ]:
import torch

scores = torch.tensor([8.0, 3.0, 1.0, -2.0])
for temperature in (1, 4):
    probabilities = torch.softmax(scores / temperature, dim=0)
    print(f"T={temperature}:", probabilities.round(decimals=3).tolist())

**My observation:** At T=1 the first answer dominates. At T=4 the second answer is easier to notice. That extra information might help, but only if the teacher has learned something useful that the small model has missed.

## 3. Compare the three trained models

`train.py` trains an ordinary student from the labels, a larger teacher, and a new student from both labels and teacher scores. It picks a model using validation, then reports the final held-out test. Here I read its saved results; I do not retrain inside the notebook.

In [ ]:
import json

report_path = root / "artifacts/results.json"
report = json.loads(report_path.read_text())
print("Train / validation / test rows:", report["split_rows"])
print("Selected before final test:", report["selected_classifier"])
print("Encoder:", report["encoder"])

In [ ]:
for name in ("ordinary_student", "teacher", "distilled_student"):
    clean = report["test"]["original"][name]["accuracy"]
    missing = report["test"]["missing_letter"][name]["accuracy"]
    swapped = report["test"]["swapped_letters"][name]["accuracy"]
    print(f"{name:19} clean {clean:.2%} | missing letter {missing:.2%} | swapped letters {swapped:.2%}")

**What I found:** The teacher got the highest clean test score. The distilled student did worse than the ordinary student on all three test checks. I kept the ordinary student because it was chosen from validation before the test was opened. The typo checks are generated examples, not a real-world typo benchmark.

## 4. How fast is the whole request?

The student classifier is tiny, but the text encoder must run first. That full path is the number a user would feel.

In [ ]:
for name in ("ordinary_student", "teacher", "distilled_student"):
    size = report["sizes"][name]
    time = report["timings"][name]
    print(f"{name:19} {size['parameters']:>7,} classifier parameters | "
          f"classifier {time['classifier_p50_ms']:.4f} ms | "
          f"text to intent {time['full_p50_ms']:.3f} ms")

**Verdict:** The small classifier works well on this Bitext split, but this distillation setup did not improve it. Full text-to-intent time was about 2.3 ms on the measured CPU. Related templates in Bitext and very small counts for some intents mean this is still an experiment, not proof that it will work on new customer messages. The next useful test would use independently collected, labeled support requests.